# 04 — Propensity Model Training

## Objective
Train and evaluate a propensity model that estimates:

> `P(conversion | customer, product, channel, context)`

The notebook consumes the prepared dataset produced by notebook 03 and keeps the same **time-aware evaluation** logic.


## 1. Install dependencies

Use a compact modeling stack for the MVP. We start with an interpretable Logistic Regression baseline and a stronger tree-based model.


In [1]:
!pip -q install pyarrow scikit-learn joblib


## 2. Imports and configuration


In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from google.colab import drive
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    classification_report,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

RANDOM_STATE = 42
TARGET = "had_conversion"


## 3. Load the prepared decision dataset

Notebook 03 saves the dataset persistently in Google Drive, so we do **not** rebuild the 1,083 raw send files here.


In [3]:
drive.mount("/content/drive", force_remount=False)

PROJECT_DIR = Path("/content/drive/MyDrive/03 Projects/Factored Hackathon")
DATASET_PATH = PROJECT_DIR / "data/processed/decision_dataset_base.parquet"
ARTIFACT_DIR = PROJECT_DIR / "artifacts/propensity"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}. Run notebook 03 cell 14 first."

df = pd.read_parquet(DATASET_PATH)
print("Shape:", df.shape)
print("Date range:", df["send_date"].min(), "→", df["send_date"].max())
print("Conversion rate:", f"{df[TARGET].mean():.4%}")


Mounted at /content/drive
Shape: (1214030, 33)
Date range: 2023-07-01 06:00:42 → 2026-06-18 05:59:53
Conversion rate: 0.7282%


## 4. Define model features

These are the same 22 decision-time features defined in notebook 03. Outcome fields such as opens, clicks, and conversion metadata are intentionally excluded to avoid leakage.


In [4]:
NUMERIC_FEATURES = [
    "credit_score",
    "estimated_monthly_income",
    "expected_conversion_rate",
    "customer_age",
    "customer_tenure_days",
    "send_hour",
    "send_day_of_week",
    "send_month",
    "is_weekend",
]

CATEGORICAL_FEATURES = [
    "country",
    "gender",
    "segment",
    "marital_status",
    "education_level",
    "customer_status",
    "accepts_marketing",
    "campaign_type",
    "campaign_objective",
    "promoted_product",
    "target_segment",
    "target_country",
    "send_channel",
]

MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

missing = [c for c in MODEL_FEATURES + [TARGET, "send_date"] if c not in df.columns]
assert not missing, f"Missing expected columns: {missing}"

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total model features:", len(MODEL_FEATURES))


Numeric features: 9
Categorical features: 13
Total model features: 22


## 5. Recreate the temporal train / validation / test split

We preserve the same chronological 70% / 15% / 15% split as notebook 03. The model learns from older observations and is evaluated on later ones.


In [5]:
df = df.sort_values("send_date").reset_index(drop=True)

train_cutoff = df["send_date"].quantile(0.70)
validation_cutoff = df["send_date"].quantile(0.85)

train_df = df[df["send_date"] <= train_cutoff].copy()
validation_df = df[(df["send_date"] > train_cutoff) & (df["send_date"] <= validation_cutoff)].copy()
test_df = df[df["send_date"] > validation_cutoff].copy()

for name, split in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(f"{name:10s} rows={len(split):,} conversion_rate={split[TARGET].mean():.4%}")

X_train, y_train = train_df[MODEL_FEATURES], train_df[TARGET].astype(int)
X_val, y_val = validation_df[MODEL_FEATURES], validation_df[TARGET].astype(int)
X_test, y_test = test_df[MODEL_FEATURES], test_df[TARGET].astype(int)


Train      rows=849,821 conversion_rate=0.7324%
Validation rows=182,104 conversion_rate=0.7457%
Test       rows=182,105 conversion_rate=0.6908%


## 6. Build preprocessing pipeline

- Numeric variables: median imputation + scaling.
- Categorical variables: most-frequent imputation + one-hot encoding.

The preprocessing lives inside the model pipeline so training and inference use exactly the same transformations.


In [16]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_FEATURES),
    ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
])


## 7. Train an interpretable baseline — Logistic Regression

Because positive conversions are rare (~0.7%), we use class weighting and judge the model mainly with **PR-AUC (Average Precision)** rather than accuracy.


In [21]:
logistic_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1,
    )),
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained.")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


max_iter reached after 1504 seconds
Logistic Regression trained.


## 8. Evaluate the baseline on validation


In [23]:
def evaluate_probabilities(name, y_true, probabilities):
    pr_auc = average_precision_score(y_true, probabilities)
    roc_auc = roc_auc_score(y_true, probabilities)
    print(f"{name}")
    print(f"  PR-AUC / Average Precision: {pr_auc:.6f}")
    print(f"  ROC-AUC:                    {roc_auc:.6f}")
    return {"model": name, "pr_auc": pr_auc, "roc_auc": roc_auc}

val_prob_logistic = logistic_model.predict_proba(X_val)[:, 1]
logistic_metrics = evaluate_probabilities("Logistic Regression", y_val, val_prob_logistic)


Logistic Regression
  PR-AUC / Average Precision: 0.007574
  ROC-AUC:                    0.494867


## 9. Inspect lift by propensity decile

For Next Best Action, ranking quality matters: customers/actions with higher predicted propensity should convert more often than low-scored ones.


In [25]:
def propensity_deciles(y_true, probabilities, n_bins=10):
    tmp = pd.DataFrame({"actual": np.asarray(y_true), "propensity": probabilities})
    tmp["decile"] = pd.qcut(tmp["propensity"], q=n_bins, labels=False, duplicates="drop") + 1
    result = (
        tmp.groupby("decile", observed=True)
        .agg(rows=("actual", "size"), conversions=("actual", "sum"), observed_conversion_rate=("actual", "mean"), avg_propensity=("propensity", "mean"))
        .sort_index(ascending=False)
    )
    return result

val_deciles = propensity_deciles(y_val, val_prob_logistic)
display(val_deciles)


,rows,conversions,observed_conversion_rate,avg_propensity
decile,,,,
10,18211,145,0.007962,0.999829
9,18210,125,0.006864,0.995148
8,18210,117,0.006425,0.954236
7,18211,148,0.008127,0.764083
6,18210,142,0.007798,0.377743
5,18210,130,0.007139,0.100629
4,18211,132,0.007248,0.018442
3,18210,128,0.007029,0.002545
2,18210,151,0.008292,0.000203


## 10. Select an operating threshold only if needed

The Next Best Action layer will mainly use probabilities as scores. A hard classification threshold is not required for ranking, but this cell helps inspect precision/recall trade-offs.


In [28]:
precision, recall, thresholds = precision_recall_curve(y_val, val_prob_logistic)
threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
})

# Example: threshold with highest F1 on validation (diagnostic only)
f1 = 2 * threshold_table["precision"] * threshold_table["recall"] / (threshold_table["precision"] + threshold_table["recall"] + 1e-12)
best_idx = f1.idxmax()
best_threshold = float(threshold_table.loc[best_idx, "threshold"])

print("Diagnostic best-F1 threshold:", best_threshold)
print(threshold_table.loc[best_idx])


Diagnostic best-F1 threshold: 0.9985764032257011
threshold    0.998576
precision    0.008205
recall       0.129602
Name: 160647, dtype: float64


## 11. Final evaluation on test

Only after the modeling choices are fixed using validation do we evaluate once on the most recent test period.


In [29]:
test_prob_logistic = logistic_model.predict_proba(X_test)[:, 1]
test_metrics = evaluate_probabilities("Logistic Regression — TEST", y_test, test_prob_logistic)

test_deciles = propensity_deciles(y_test, test_prob_logistic)
display(test_deciles)


Logistic Regression — TEST
  PR-AUC / Average Precision: 0.007156
  ROC-AUC:                    0.516547


,rows,conversions,observed_conversion_rate,avg_propensity
decile,,,,
10,18211,121,0.006644,0.999995
9,18210,145,0.007963,0.999886
8,18211,134,0.007358,0.998940
7,18210,138,0.007578,0.993219
6,18210,133,0.007304,0.963255
5,18211,125,0.006864,0.832204
4,18210,109,0.005986,0.475167
3,18211,122,0.006699,0.120821
2,18210,113,0.006205,0.011993


## 12. Save the trained model and metadata

The complete sklearn pipeline is saved so notebook 05 can score candidate Product × Channel actions without rebuilding preprocessing manually.


In [30]:
MODEL_PATH = ARTIFACT_DIR / "propensity_logistic_pipeline.joblib"
METADATA_PATH = ARTIFACT_DIR / "propensity_model_metadata.json"

joblib.dump(logistic_model, MODEL_PATH)

metadata = {
    "target": TARGET,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "model_features": MODEL_FEATURES,
    "train_cutoff": str(train_cutoff),
    "validation_cutoff": str(validation_cutoff),
    "validation_metrics": logistic_metrics,
    "test_metrics": test_metrics,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Model saved to:", MODEL_PATH)
print("Metadata saved to:", METADATA_PATH)


Model saved to: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/propensity_logistic_pipeline.joblib
Metadata saved to: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/propensity_model_metadata.json


## 13. Produce scored test rows for the Next Best Action layer

These scores are **propensity estimates**, not yet the final action. Notebook 05 will combine them with product/channel candidates, send cost, expected conversion value, and business rules.


In [31]:
scored_test = test_df[[
    "customer_id",
    "campaign_id",
    "send_date",
    "country",
    "promoted_product",
    "send_channel",
    TARGET,
    "send_cost",
    "conversion_value",
]].copy()

scored_test["predicted_conversion_probability"] = test_prob_logistic

SCORED_PATH = ARTIFACT_DIR / "propensity_scored_test.parquet"
scored_test.to_parquet(SCORED_PATH, index=False)

print("Scored rows:", f"{len(scored_test):,}")
print("Saved to:", SCORED_PATH)
display(scored_test.head())


Scored rows: 182,105
Saved to: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/propensity_scored_test.parquet


,customer_id,campaign_id,send_date,country,promoted_product,send_channel,had_conversion,send_cost,conversion_value,predicted_conversion_probability
1031925,CLI-GNKSEFE9FWEN,CMP-LIEBCB6I4IWC,2026-01-15 19:13:29,México,Tarjeta Crédito,Push,0,0.0002,0.0,0.999813
1031926,CLI-9F8999U5NNLP,CMP-173ISW5CHVBB,2026-01-15 19:13:59,México,Préstamo Personal,Email,0,0.0039,0.0,0.077361
1031927,CLI-486AFWFW38JL,CMP-173ISW5CHVBB,2026-01-15 19:17:28,México,Préstamo Personal,Email,0,0.0039,0.0,0.946937
1031928,CLI-OZ7SV7QCUPVP,CMP-LIEBCB6I4IWC,2026-01-15 19:18:46,Argentina,Tarjeta Crédito,Push,0,0.0000,0.0,0.997929
1031929,CLI-97JLTJS1SPNQ,CMP-OZ3ECV86GFA8,2026-01-15 19:19:10,México,Tarjeta Crédito,Email,0,0.0000,0.0,0.999814


## Next step

Notebook **05 — Next Best Action** should:

1. generate eligible Product × Channel candidate actions for a customer;
2. score every candidate with this propensity pipeline;
3. combine `predicted_conversion_probability`, send cost and expected conversion value;
4. apply business constraints and eligibility rules;
5. let the agent explain or choose the recommended next action.
